In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!pip install editdistance opencv-python-headless -q

In [ ]:
!pip install datasets -q

In [ ]:
from datasets import load_dataset

iam_dataset = load_dataset("Teklia/IAM-line")
print(iam_dataset)
print(iam_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 6482
    })
    validation: Dataset({
        features: ['image', 'text'],
        num_rows: 976
    })
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 2915
    })
})
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=L size=2467x128 at 0x7FE31C79AC60>, 'text': 'put down a resolution on the subject'}


In [ ]:
import os

# Convert to a flat list of (PIL Image, transcription) pairs
train_samples_raw = [(item["image"], item["text"]) for item in iam_dataset["train"]]
val_samples_raw = [(item["image"], item["text"]) for item in iam_dataset["validation"]]

all_characters = set()
for _, transcription in train_samples_raw + val_samples_raw:
    all_characters.update(transcription)

charset = sorted(all_characters)
print(f"Charset size: {len(charset)}")
print("Charset:", "".join(charset))

Charset size: 79
Charset:  !"#&'()*+,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [ ]:
import json

os.makedirs("/content/drive/MyDrive/note2snap_crnn", exist_ok=True)
charset_path = "/content/drive/MyDrive/note2snap_crnn/charset.json"
with open(charset_path, "w") as f:
    json.dump(charset, f)

print("Saved charset to", charset_path)

Saved charset to /content/drive/MyDrive/note2snap_crnn/charset.json


In [ ]:
import numpy as np

IMG_HEIGHT = 32
IMG_WIDTH = 256

def preprocess_pil_image(pil_image):
    grayscale = pil_image.convert("L")  # matches Android's grayscale preprocessing
    resized = grayscale.resize((IMG_WIDTH, IMG_HEIGHT))
    array = np.array(resized, dtype=np.float32) / 255.0
    return array[..., np.newaxis]  # add channel dimension -> (H, W, 1)

# Pre-convert everything once — fine for IAM-line's size (~7,400 images total).
train_images_np = np.stack([preprocess_pil_image(img) for img, _ in train_samples_raw])
val_images_np = np.stack([preprocess_pil_image(img) for img, _ in val_samples_raw])

train_samples = list(zip(train_images_np, [t for _, t in train_samples_raw]))
val_samples = list(zip(val_images_np, [t for _, t in val_samples_raw]))

In [ ]:
import tensorflow as tf
import numpy as np # Ensure numpy is imported for np.array

def encode_label(transcription, max_label_length=64):
    indices = [char_to_index[char] for char in transcription if char in char_to_index]
    length = len(indices)
    padded = indices + [0] * (max_label_length - length) # Padding with 0 for blank
    return np.array(padded[:max_label_length], dtype=np.int32), min(length, max_label_length)


def build_dataset(samples, batch_size=32, shuffle=True):
    images = np.stack([s[0] for s in samples])       # s[0] = pixel array (already preprocessed in Step 3)
    transcriptions = [s[1] for s in samples]          # s[1] = the text label

    encoded_labels = []
    label_lengths = []
    for t in transcriptions:
        encoded, length = encode_label(t)
        encoded_labels.append(encoded)
        label_lengths.append(length)

    # Images are already numpy arrays at this point, so we load them straight
    # in — no file-reading/decoding step needed here.
    image_ds = tf.data.Dataset.from_tensor_slices(images)

    label_ds = tf.data.Dataset.from_tensor_slices(np.array(encoded_labels))
    label_length_ds = tf.data.Dataset.from_tensor_slices(np.array(label_lengths, dtype=np.int32))

    dataset = tf.data.Dataset.zip((image_ds, label_ds, label_length_ds))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000, seed=42)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


char_to_index = {char: index + 1 for index, char in enumerate(charset)}  # 0 reserved for CTC blank
index_to_char = {index: char for char, index in char_to_index.items()}

train_dataset = build_dataset(train_samples, batch_size=32, shuffle=True)
val_dataset = build_dataset(val_samples, batch_size=32, shuffle=False)

print("Train batches:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Val batches:", tf.data.experimental.cardinality(val_dataset).numpy())

Train batches: 203
Val batches: 31


In [ ]:
for images, labels, label_lengths in train_dataset.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("First label indices:", labels[0].numpy())
    print("First label length:", label_lengths[0].numpy())
    decoded = "".join(index_to_char[i] for i in labels[0].numpy() if i != 0)
    print("Decoded back to text:", decoded)

Image batch shape: (32, 32, 256, 1)
Label batch shape: (32, 64)
First label indices: [72 73 74 57 78 62 67 60  1 73 61 58 66  1 73 68 57 54 78  1 13  1 47 61
 58  1 56 68 67 59 58 71 58 67 56 58  1 76 62 65 65  1 66 58 58 73  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
First label length: 46
Decoded back to text: studying them today . The conference will meet


In [ ]:
from tensorflow.keras import layers, Model

def build_crnn_model(img_height, img_width, num_classes):
    inputs = layers.Input(shape=(img_height, img_width, 1), name="image_input")

    # --- Convolutional feature extractor ---
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)          # height 32 -> 16, width 256 -> 128

    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2))(x)          # height 16 -> 8, width 128 -> 64

    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 1))(x)          # height 8 -> 4, width unchanged (preserve sequence length)

    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 1))(x)          # height 4 -> 2, width unchanged

    # Collapse the (now-small) height dimension into the channel dimension,
    # turning the 2D feature map into a 1D sequence along the width axis —
    # each "time step" the RNN sees corresponds to a vertical strip of the image.
    new_shape = (img_width // 4, (img_height // 16) * 128) # Corrected height calculation (32 -> 2)
    x = layers.Reshape(target_shape=new_shape)(x)
    x = layers.Dense(64, activation="relu")(x)

    # --- Recurrent sequence layers ---
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25))(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.25))(x)

    # --- Output layer: one softmax distribution per time-step ---
    # num_classes already includes the CTC blank at index 0.
    # Change activation to None to output logits for CTC loss
    outputs = layers.Dense(num_classes, name="char_probabilities")(x)

    return Model(inputs=inputs, outputs=outputs, name="note2snap_crnn")

num_classes = len(charset) + 1  # +1 for CTC blank at index 0
model = build_crnn_model(IMG_HEIGHT, IMG_WIDTH, num_classes)
model.summary()

Model: "note2snap_crnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image_input (InputLayer)        │ (None, 32, 256, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 32, 256, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 16, 128, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 16, 128, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 8, 64, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 8, 64, 128)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 4, 64, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 4, 64, 128)     │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 4, 64, 128)     │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 2, 64, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_3 (Reshape)             │ (None, 64, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64, 64)         │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 64, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 64, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ char_probabilities (Dense)      │ (None, 64, 80)         │        10,320 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 629,520 (2.40 MB)

 Trainable params: 629,264 (2.40 MB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

class CTCLayer(layers.Layer):
    """Computes CTC loss during training and passes predictions through unchanged."""

    def __init__(self, name=None):
        super().__init__(name=name)

    def call(self, labels, logits, label_length):
        # actual_batch_size is determined by the first dimension of labels
        actual_batch_size = tf.shape(labels)[0]

        # time_steps is the sequence length from the logits
        original_time_steps = tf.shape(logits)[1]

        # logit_length for each sample is the max_time from the logits
        logit_length = tf.fill([actual_batch_size], original_time_steps)

        # label_length is already (actual_batch_size,)

        # --- Convert dense labels to SparseTensor ---
        max_label_length = tf.shape(labels)[1]

        mask = tf.sequence_mask(label_length, maxlen=max_label_length)
        sparse_indices = tf.where(mask)
        sparse_values = tf.gather_nd(labels, sparse_indices)
        dense_shape = tf.cast(tf.shape(labels), tf.int64)
        sparse_labels = tf.SparseTensor(sparse_indices, sparse_values, dense_shape)

        loss = tf.nn.ctc_loss(
            labels=sparse_labels,
            logits=logits, # Pass original (batch-major) logits
            label_length=label_length,
            logit_length=logit_length,
            blank_index=0
            # time_major argument removed as it's deprecated and implicitly handled
            # tf.nn.ctc_loss expects batch-major logits by default in this TF version
        )
        self.add_loss(tf.reduce_mean(loss))
        return logits # Return original (batch-major) logits for model summary/inference


def build_training_model(base_model, max_label_length=64):
    label_input = layers.Input(name="label_input", shape=(max_label_length,), dtype="int32")
    label_length_input = layers.Input(name="label_length_input", shape=(), dtype="int32")

    predictions = base_model.output # These are now logits (batch-major)
    ctc_output = CTCLayer(name="ctc_loss")(label_input, predictions, label_length_input)

    training_model = Model(
        inputs=[base_model.input, label_input, label_length_input],
        outputs=ctc_output
    )
    return training_model


training_model = build_training_model(model)
training_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3))
training_model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 32, 256,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 32, 256,   │        320 │ image_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_12    │ (None, 16, 128,   │          0 │ conv2d_12[0][0]   │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 16, 128,   │     18,496 │ max_pooling2d_12… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_13    │ (None, 8, 64, 64) │          0 │ conv2d_13[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 8, 64,     │     73,856 │ max_pooling2d_13… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_14    │ (None, 4, 64,     │          0 │ conv2d_14[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 4, 64,     │    147,584 │ max_pooling2d_14… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4, 64,     │        512 │ conv2d_15[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_15    │ (None, 2, 64,     │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 64, 256)   │          0 │ max_pooling2d_15… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64, 64)    │     16,448 │ reshape_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 64, 256)   │    197,632 │ dense_2[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 64, 128)   │    164,352 │ bidirectional_4[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ label_input         │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ char_probabilities  │ (None, 64, 80)    │     10,320 │ bidirectional_5[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ label_length_input  │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 629,520 (2.40 MB)

 Trainable params: 629,264 (2.40 MB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
def to_training_inputs(images, labels, label_lengths):
    return (
        {"image_input": images, "label_input": labels, "label_length_input": label_lengths},
        labels  # dummy target; CTCLayer computes the real loss internally via add_loss
    )

train_dataset_for_training = train_dataset.map(to_training_inputs)
val_dataset_for_training = val_dataset.map(to_training_inputs)

In [ ]:
checkpoint_path = "/content/drive/MyDrive/note2snap_crnn/baseline_checkpoint.weights.h5"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        checkpoint_path, save_weights_only=True, save_best_only=True, monitor="val_loss"
    ),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

history = training_model.fit(
    train_dataset_for_training,
    validation_data=val_dataset_for_training,
    epochs=40,
    callbacks=callbacks
)

Epoch 1/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - loss: 136.1603 - val_loss: 152.6020
Epoch 2/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 131.3679 - val_loss: 142.5974
Epoch 3/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - loss: 130.3595 - val_loss: 178.7493
Epoch 4/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - loss: 129.3065 - val_loss: 131.1197
Epoch 5/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - loss: 126.2744 - val_loss: 127.9584
Epoch 6/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - loss: 123.4436 - val_loss: 124.3678
Epoch 7/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 121.2079 - val_loss: 127.7798
Epoch 8/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - loss: 119.1041 - val_loss: 121.2460
Epoch 9/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 117.0902 - val_loss: 132.3483
Epoch 10/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - loss: 115.1648 - val_loss: 117.2100
Epoch 11/40
203/203 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - loss: 113.1226 - val_loss: 136

In [ ]:
import editdistance
import numpy as np

def greedy_decode_predictions(predictions, index_to_char):
    """Mirrors the same greedy CTC decoding logic as the Kotlin CrnnCtcDecoder."""
    decoded_texts = []
    predicted_classes = np.argmax(predictions, axis=-1)  # (batch, time_steps)

    for sequence in predicted_classes:
        text = []
        previous_class = -1
        for class_index in sequence:
            is_blank = class_index == 0
            if not is_blank and class_index != previous_class:
                text.append(index_to_char.get(int(class_index), ""))
            previous_class = class_index
        decoded_texts.append("".join(text))

    return decoded_texts


def evaluate_cer(base_model, dataset, index_to_char, max_batches=None):
    total_edit_distance = 0
    total_reference_length = 0
    examples_shown = 0

    for batch_index, (images, labels, label_lengths) in enumerate(dataset):
        if max_batches is not None and batch_index >= max_batches:
            break

        predictions = base_model.predict(images, verbose=0)
        predicted_texts = greedy_decode_predictions(predictions, index_to_char)

        for i in range(len(predicted_texts)):
            true_length = int(label_lengths[i].numpy())
            true_indices = labels[i].numpy()[:true_length]
            true_text = "".join(index_to_char.get(int(idx), "") for idx in true_indices)

            distance = editdistance.eval(predicted_texts[i], true_text)
            total_edit_distance += distance
            total_reference_length += max(len(true_text), 1)

            if examples_shown < 5:
                print(f"  Predicted: {predicted_texts[i]!r}")
                print(f"  Actual:    {true_text!r}")
                print()
                examples_shown += 1

    cer = total_edit_distance / total_reference_length
    return cer


print("Sample predictions vs. ground truth:")
cer_score = evaluate_cer(model, val_dataset, index_to_char, max_batches=20)
print(f"Validation CER: {cer_score:.4f} ({cer_score * 100:.2f}%)")

Sample predictions vs. ground truth:
  Predicted: 'the aoe an etind maettteted of that'
  Actual:    'It was a splendid interpretation of the'

  Predicted: 'cerettitet the the danmeranen an rtien'
  Actual:    'sympathetic C O . Paul Daneman gave another'

  Predicted: 'ad he wad of the waen ad waed arteen .'
  Actual:    'part . The rest of the cast were well chosen ,'

  Predicted: 'wite thrmen iaeritile mentine an wane hant o the'
  Actual:    'with James Maxwell making a fine job of the'

  Predicted: '"  t  . \''
  Actual:    '" The Little Key . "'

Validation CER: 0.6413 (64.13%)
